## Lab 04 – Materialized Views in Snowflake
**Snowflake Fundamentals 4-day class Lab · © 2026 Innovation In Software Corporation. All rights reserved.**

Topics covered:
1. Creating a Materialized View over a source table
2. Automatic incremental refresh — no manual refresh needed
3. Query performance — precomputed aggregations vs. source table scans
4. Observing refresh history via MATERIALIZED_VIEW_REFRESH_HISTORY()
5. DML impact — UPDATE and DELETE propagation to the MV
6. Limitations — subqueries in WHERE clause not supported

### DEMO 1 | Context Setup

> **[INSTRUCTOR NOTE]**
> A dedicated `DEMO_DB` and warehouse are created for this lab. Unlike Dynamic Tables, Materialized Views refresh automatically in the background — there is no `ALTER ... REFRESH` command. Enterprise Edition or higher is required.

In [ ]:
-- Check current role
SELECT CURRENT_ROLE();

In [ ]:
DROP DATABASE IF EXISTS DEMO_DB;

In [ ]:
CREATE DATABASE IF NOT EXISTS DEMO_DB;
CREATE SCHEMA IF NOT EXISTS DEMO_DB.DEMO_SCHEMA;
USE DATABASE DEMO_DB;
USE SCHEMA DEMO_SCHEMA;
CREATE WAREHOUSE IF NOT EXISTS DEMO_WH WITH WAREHOUSE_SIZE = 'XSMALL' AUTO_SUSPEND = 300;
USE WAREHOUSE DEMO_WH;

### DEMO 2 | Create Source Table and Materialized View

> **[INSTRUCTOR NOTE]**
> The MV aggregates `SUM(amount)` by `product_name`. Snowflake maintains the MV incrementally — when rows in `sales` change, only the affected partitions are recomputed, not the entire dataset.

In [ ]:
-- Create a source table to store sales data
CREATE OR REPLACE TABLE sales (
    sale_id      INT,
    product_name STRING,
    amount       DECIMAL(10,2),
    sale_date    DATE
);

In [ ]:
-- Create a materialized view that aggregates sales by product
CREATE OR REPLACE MATERIALIZED VIEW sales_by_product
AS
    SELECT product_name, SUM(amount) AS total_sales
    FROM sales
    GROUP BY product_name;

### DEMO 3 | Insert Data and Query the Materialized View

> **[INSTRUCTOR NOTE]**
> After inserting rows into `sales`, the MV refreshes automatically. Querying `sales_by_product` returns the precomputed aggregation — Snowflake may serve the result directly from the MV without scanning `sales` at all.
>
> Materialized views refresh incrementally when the source table changes, managed by Snowflake’s background services — no manual refresh needed, unlike dynamic tables.

In [ ]:
INSERT INTO sales VALUES
    (1, 'Laptop', 1000.00, '2025-09-20'),
    (2, 'Laptop', 1500.00, '2025-09-20'),
    (3, 'Phone',   800.00, '2025-09-21'),
    (4, 'Phone',   900.00, '2025-09-21');

In [ ]:
SELECT * FROM sales_by_product;

### DEMO 4 | Explore Materialized View Behaviour

> **[INSTRUCTOR NOTE]**
> `MATERIALIZED_VIEW_REFRESH_HISTORY()` records each background refresh run. After DML on `sales`, wait 10–30 seconds before querying the MV — Snowflake will have applied the incremental refresh automatically.

In [ ]:
SELECT *
FROM TABLE(INFORMATION_SCHEMA.MATERIALIZED_VIEW_REFRESH_HISTORY())
WHERE MATERIALIZED_VIEW_NAME = 'SALES_BY_PRODUCT';

In [ ]:
-- Test Source Table Modifications
UPDATE sales SET amount = 1200.00 WHERE sale_id = 1;
DELETE FROM sales WHERE sale_id = 2;

In [ ]:
-- Wait briefly (e.g. 10–30 seconds) for Snowflake to refresh the MV, then query it
SELECT * FROM sales_by_product;

### DEMO 5 | Materialized View Limitations

> **[INSTRUCTOR NOTE]**
> Materialized Views do not support subqueries in the WHERE clause, non-deterministic functions, or joins (in most editions). The statement below will fail — show students the error message and explain why.

In [ ]:
-- This will fail — subqueries in WHERE are not supported in Materialized Views
CREATE OR REPLACE MATERIALIZED VIEW invalid_mv
AS
    SELECT product_name, SUM(amount) AS total_sales
    FROM sales
    WHERE amount > (SELECT AVG(amount) FROM sales)
    GROUP BY product_name;

### DEMO CLEANUP

> Drop objects created during the demo.

In [ ]:
-- DROP DATABASE IF EXISTS DEMO_DB;   -- uncomment if you are not continuing to the next lab
-- DROP WAREHOUSE IF EXISTS DEMO_WH;  -- uncomment if you are not continuing to the next lab
USE WAREHOUSE COMPUTE_WH;